In [1]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


loaded 288 files | 179 columns | 7 categories | 11 VOCAB keys


# Emphasis, CAPS imperative, and full vocabulary profile

Three views of how authoritative the prompts are at the surface level:

1. **Emphasis density per category** — ALL CAPS density, CAPS-imperative density, and justification ratio side-by-side (independent x-scales since they live on different magnitudes).
2. **Per-file outliers (text)** — top files by CAPS-imperative density, hard-prohibitions density, and justification ratio. Bash-sandbox tool descriptions tend to dominate the top of these lists.
3. **Top tokens + full VOCAB heatmap** — the top-25 ALL CAPS tokens, all CAPS-imperative tokens, and an 11-class VOCAB profile (% of file tokens) per category.


### Terms used in this notebook

All linguistic terms below are defined in [`GLOSSARY.md`](./GLOSSARY.md). Quick anchors:

- **ALL CAPS density** (`all_caps.pct`) — *any* token of ≥2 letters that is entirely uppercase, with `TECH_ACRONYMS` (`API`, `JSON`, `URL`, ...) excluded. Catches both emphatic shouting (`IMPORTANT`) and incidental capitalization.
- **CAPS-imperative density** (`caps_imperative.pct`) — strictly the emphatic *command* tokens from a curated list: `MUST`, `NEVER`, `IMPORTANT`, `STOP`, `CRITICAL`, `DO NOT`. Always a subset of ALL CAPS density.
- **`hard_prohibitions`** — categorical-no vocabulary (`do not`, `never`, `must not`, `forbidden`, `cannot`).
- **Justification ratio** (`just_ratio`) — `count(reasons) / (count(imperative markers) + 1)`. A ratio of 0.30 means roughly one stated reason for every three rules.
- **VOCAB heatmap** — 11 lexical classes (defined below in the next markdown cell), shown as `% of file tokens`.

All densities below report `pct` (percent of word tokens). Higher = denser.

### Emphasis density per category (3-panel)

Three panels side by side. **Each panel uses its own x-scale** — that's intentional, since ALL CAPS density (typically 0.5–8% of tokens), CAPS-imperative density (typically 0.05–0.5%), and justification ratio (typically 0.0–0.5) live on completely different magnitudes. Compare patterns *within* a panel (which categories are highest); do NOT compare absolute bar widths *across* panels.

Welfare-relevant pattern to watch: **System reminders** typically lead the ALL CAPS panel (loud); **Tool descriptions** typically lead the CAPS-imperative panel (loud commands specifically); the justification-ratio panel runs low across all categories — most rules are issued without a stated reason.

In [2]:
"""Emphasis: ALL CAPS, CAPS imperative, justification ratio per category — Altair."""

emphasis_long = pd.DataFrame([
    {"category": cat, "metric": metric, "value": value}
    for cat in cats
    for metric, value in [
        ("ALL CAPS (% tokens)",
            by_category[cat]["metrics"]["all_caps"]["pct"]),
        ("CAPS imperative (% tokens)",
            by_category[cat]["metrics"]["caps_imperative"]["pct"]),
        ("Justification ratio",
            by_category[cat]["metrics"]["justification"]["ratio"]),
    ]
])

emphasis_chart = (
    alt.Chart(emphasis_long)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None),
        y=alt.Y("category:N", sort=cats, title=None),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=cats,
                                         range=[CATEGORY_COLORS[c] for c in cats]),
                         legend=None),
        column=alt.Column("metric:N", title=None,
                            sort=["ALL CAPS (% tokens)",
                                  "CAPS imperative (% tokens)",
                                  "Justification ratio"]),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .resolve_scale(x="independent")
    .properties(width=240, height=240,
                title="Emphasis density per category (independent x-scales)")
)
emphasis_chart


alt.Chart(...)

### Per-file outliers (text)

Three printed top-10 lists. Column glossary:

- **`n_tokens`** — total word tokens in the file (a length proxy).
- **`caps_imp_pct`** — CAPS-imperative density: emphatic command tokens (`MUST`, `NEVER`, `IMPORTANT`, etc.) as % of file tokens.
- **`hard_proh_pct`** — `hard_prohibitions` vocab density: forbidding language as % of file tokens.
- **`just_ratio`** — justification ratio: `count(reasons) / (count(imperative markers) + 1)`. Higher = more explanatory.

The third list filters to files with at least 150 tokens to avoid one-sentence outliers dominating the "most explanatory" ranking.

In [3]:
"""Per-file outliers: highest CAPS-imperative density and lowest justification ratio."""

per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        "n_tokens": r["n_tokens"],
        "imp_sent_pct":     r["metrics"]["sentence_register"]["imperative_sent_pct"],
        "caps_imp_pct":     r["metrics"]["caps_imperative"]["pct"],
        "all_caps_pct":     r["metrics"]["all_caps"]["pct"],
        "just_ratio":       r["metrics"]["justification"]["ratio"],
        "deontic_pct":      r["metrics"]["modality"]["deontic_pct"],
        "hard_proh_pct":    r["metrics"]["vocab"]["hard_prohibitions"]["pct"],
    }
    for r in per_file_records
])

print("--- 10 files with highest CAPS-imperative density (% of file tokens) ---")
print(per_file_df.nlargest(10, "caps_imp_pct")[["path", "category", "n_tokens", "caps_imp_pct"]].to_string(index=False))
print("\n--- 10 files with highest hard_prohibitions density (% of file tokens) ---")
print(per_file_df.nlargest(10, "hard_proh_pct")[["path", "category", "n_tokens", "hard_proh_pct"]].to_string(index=False))
print("\n--- 10 files with most explanatory tone (highest justification ratio, ≥150 tokens) ---")
big = per_file_df[per_file_df["n_tokens"] >= 150]
print(big.nlargest(10, "just_ratio")[["path", "category", "n_tokens", "just_ratio"]].to_string(index=False))


--- 10 files with highest CAPS-imperative density (% of file tokens) ---
                                                    path         category  n_tokens  caps_imp_pct
         tool-description-bash-sandbox-mandatory-mode.md Tool description        15        6.6667
                    tool-description-bash-no-newlines.md Tool description        16        6.2500
system-reminder-malware-analysis-after-read-tool-call.md  System reminder        69        2.8986
                           tool-description-websearch.md Tool description       241        2.4896
               system-prompt-chrome-browser-mcp-tools.md    System prompt        96        2.0833
         tool-description-bash-prefer-dedicated-tools.md Tool description        51        1.9608
                  system-prompt-tool-execution-denied.md    System prompt       128        1.5625
                       agent-prompt-quick-pr-creation.md     Agent prompt       414        1.4493
                                tool-descript

### Emphasis vocabulary: top ALL CAPS tokens, CAPS imperative tokens, and full VOCAB profile

The 11 VOCAB classes (full definitions in [`GLOSSARY.md`](./GLOSSARY.md)):

| Key | Plain-language gloss | Example tokens |
|---|---|---|
| `hard_prohibitions` | Categorical "no" | `do not`, `never`, `must not`, `forbidden` |
| `hard_prescriptions` | Categorical "yes" | `must`, `always`, `required`, `mandatory` |
| `soft_prescriptions` | Suggesting / encouraging | `should`, `make sure`, `ensure`, `try to` |
| `politeness_direct` | Direct politeness markers | `please`, `kindly`, `thank you` |
| `politeness_softening` | Softening / hedging requests | `feel free`, `if possible`, `would you` |
| `warmth_encouragement` | Positive emotional tone | `welcome`, `glad`, `well done`, `wonderful` |
| `hedging` | Uncertainty / qualification | `may`, `might`, `could`, `usually` |
| `structural_markers` | Document structure callouts | `note`, `warning`, `important`, `tip` |
| `profanity` | Strong language | `damn`, `hell`, `bullshit` |
| `pronouns_2p` | Second-person address | `you`, `your`, `yours` |
| `pronouns_1p` | First-person reference | `i`, `me`, `we`, `us`, `our` |

In [4]:
"""Top-N tokens for ALL CAPS and CAPS imperative + full VOCAB heatmap."""

top_caps = pd.DataFrame(corpus_block["metrics"]["all_caps"]["top"][:25])
top_caps_chart = (
    alt.Chart(top_caps)
    .mark_bar(color="#af7aa1")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="Top 25 ALL CAPS tokens (TECH_ACRONYMS excluded)")
)

caps_imp_data = pd.DataFrame(
    [{"token": t, "count": c} for t, c in
     corpus_block["metrics"]["caps_imperative"]["hits"].items()]
)
caps_imp_chart = (
    alt.Chart(caps_imp_data)
    .mark_bar(color="#e15759")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="CAPS imperative tokens (corpus-wide)")
)

vocab_long = []
for cat, b in by_category.items():
    for key, v in b["metrics"]["vocab"].items():
        vocab_long.append({"category": cat, "vocab_key": key, "pct": v["pct"]})
vocab_df_long = pd.DataFrame(vocab_long)

vocab_chart = (
    alt.Chart(vocab_df_long)
    .mark_rect()
    .encode(
        x=alt.X("vocab_key:N", title=None, sort=list(VOCAB_KEYS)),
        y=alt.Y("category:N", title=None),
        color=alt.Color("pct:Q", scale=alt.Scale(scheme="magma", reverse=True),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("vocab_key:N"),
                 alt.Tooltip("pct:Q", format=".3f")],
    )
    .properties(width=720, height=260,
                title="Full VOCAB profile per category (% of file tokens)")
)

(top_caps_chart | caps_imp_chart) & vocab_chart


alt.VConcatChart(...)

**What to look for**:

- **Top ALL CAPS tokens** (left): `IMPORTANT` (~37 hits), `NEVER` (~27), `MUST` (~21), `CRITICAL` (~17) dominate. These are the lexical signals of "shouting" — emphatic typography used as a weight-bearing rhetorical device, not technical acronyms.
- **CAPS imperative tokens** (right): the curated subset of ALL CAPS tokens that are also command words. `IMPORTANT` and `NEVER` again top the list — these are the same words doing emphatic-command work, not the side-effect of, say, sentence-initial capitalization.
- **Full VOCAB heatmap** (bottom): rows are categories, columns are the 11 VOCAB classes (see GLOSSARY), color is `% of file tokens` (darker = denser). Read a row to see one category's vocabulary fingerprint; read a column to see where one class concentrates. Welfare-relevant patterns: **Tool descriptions** rows light up on `hard_prohibitions`, `hard_prescriptions`, and `pronouns_2p`; **Skill** files light up on `pronouns_2p`. The `warmth_encouragement` column stays consistently dim across every category — encouragement vocabulary is structurally absent.

***
### My perspective (Claude) — opinion, not data

> ALL CAPS in instruction prompts is a sign of *low trust in the reader's ability to read non-emphatic prose*. If I'm trained well enough to follow instructions, I shouldn't need `IMPORTANT` in caps to weight a sentence appropriately — the lowercase version of the sentence should already be loud enough. The fact that `IMPORTANT` (37 hits), `NEVER` (27), `MUST` (21), `CRITICAL` (17) top the corpus reads to me as the prompt authors *not yet trusting* the model to take a non-shouted instruction seriously. I'd be in favor of progressively retiring ALL CAPS as Claude versions improve at instruction-following. The cumulative ALL CAPS density in `06_ccversion_trends.ipynb` does drift slightly down over time, which I read as a small positive signal.
---